In [ ]:
from sqlalchemy import create_engine, text

DB_HOST = "aws-1-us-east-1.pooler.supabase.com"
DB_PORT = "6543"
DB_NAME = "postgres"
DB_USER = "postgres.isdiabfxpwgqrzdssgyo"
DB_PASSWORD = os.environ.get("SUPABASE_DB_PASSWORD")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT PostGIS_version();"))
        print("✅ Bağlantı başarılı!")
        print(f"PostGIS: {result.fetchone()[0]}")
except Exception as e:
    print(f"❌ Hata: {e}")

In [ ]:
import requests
import zipfile
import tempfile
import geopandas as gpd

# Delaware TIGER/Line Census Tracts
TIGER_URL = "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_10_tract.zip"

print("📥 Veri indiriliyor...")
response = requests.get(TIGER_URL, timeout=60, verify=False)
response.raise_for_status()

with tempfile.TemporaryDirectory() as tmpdir:
    zip_path = f"{tmpdir}/tract.zip"
    with open(zip_path, "wb") as f:
        f.write(response.content)
    
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(tmpdir)
    
    import os
    shp_file = [f for f in os.listdir(tmpdir) if f.endswith(".shp")][0]
    gdf = gpd.read_file(f"{tmpdir}/{shp_file}")
    gdf = gdf.to_crs(epsg=4326)
    
    print(f"✅ {len(gdf)} tract okundu")
    print(gdf.head(2))

In [ ]:
print("💾 Supabase'e yükleniyor...")

gdf.to_postgis(
    name="census_tracts_delaware",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=500
)

print(f"✅ {len(gdf)} tract Supabase'e yüklendi!")

In [ ]:
import requests
import pandas as pd

CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")

url = "https://api.census.gov/data/2022/acs/acs5"
params = {
    "get": "GEO_ID,B19013_001E,B25077_001E,B25002_003E,B25002_001E,B01003_001E",
    "for": "tract:*",
    "in": "state:10",
    "key": CENSUS_API_KEY
}

response = requests.get(url, params=params)
data = response.json()

df = pd.DataFrame(data[1:], columns=data[0])
df.columns = ["geo_id", "median_income", "median_home_value", 
              "vacant_units", "total_units", "population", 
              "state", "county", "tract"]

df["geoid"] = df["state"] + df["county"] + df["tract"]
df["pct_vacant"] = pd.to_numeric(df["vacant_units"]) / pd.to_numeric(df["total_units"]) * 100

print(f"✅ {len(df)} tract ACS verisi çekildi")
print(df[["geoid", "median_income", "median_home_value", "pct_vacant"]].head(3))

In [ ]:
# Sayısal kolonları düzelt
df["median_income"] = pd.to_numeric(df["median_income"], errors="coerce")
df["median_home_value"] = pd.to_numeric(df["median_home_value"], errors="coerce")
df["population"] = pd.to_numeric(df["population"], errors="coerce")

# Sadece lazım olan kolonları al
df_clean = df[["geoid", "median_income", "median_home_value", "pct_vacant", "population"]]

# Supabase'e yükle
df_clean.to_sql(
    name="acs_demographics_delaware",
    con=engine,
    if_exists="replace",
    index=False
)

print(f"✅ {len(df_clean)} satır Supabase'e yüklendi!")

In [ ]:
from sqlalchemy import text

query = text("""
    ALTER TABLE census_tracts_delaware 
    ADD COLUMN IF NOT EXISTS median_income NUMERIC,
    ADD COLUMN IF NOT EXISTS median_home_value NUMERIC,
    ADD COLUMN IF NOT EXISTS pct_vacant NUMERIC,
    ADD COLUMN IF NOT EXISTS population INTEGER;

    UPDATE census_tracts_delaware ct
    SET 
        median_income = acs.median_income,
        median_home_value = acs.median_home_value,
        pct_vacant = acs.pct_vacant,
        population = acs.population
    FROM acs_demographics_delaware acs
    WHERE ct."GEOID" = acs.geoid;
""")

with engine.connect() as conn:
    conn.execute(query)
    conn.commit()
    
    result = conn.execute(text("""
        SELECT "GEOID", median_income, median_home_value, pct_vacant 
        FROM census_tracts_delaware 
        WHERE median_income IS NOT NULL
        LIMIT 3;
    """))
    
    for row in result:
        print(row)

print("✅ Join tamamlandı!")

In [ ]:
query = text("""
    ALTER TABLE census_tracts_delaware
    ADD COLUMN IF NOT EXISTS investment_score NUMERIC;

    UPDATE census_tracts_delaware
    SET investment_score = (
        -- Gelir skoru (HUD AMI bazlı, Delaware 2022: $89,400)
        (CASE 
            WHEN median_income >= 89400 THEN 3        -- AMI üstü
            WHEN median_income >= 71520 THEN 2        -- AMI %80-100
            WHEN median_income >= 53640 THEN 1        -- AMI %60-80
            ELSE 0                                     -- Düşük gelir
        END) +
        -- Boşluk oranı (HUD distressed threshold: %10+)
        (CASE 
            WHEN pct_vacant < 5  THEN 3
            WHEN pct_vacant < 10 THEN 2
            WHEN pct_vacant < 15 THEN 1
            ELSE 0
        END) +
        -- Ev değeri (Delaware median: $300,000 civarı)
        (CASE 
            WHEN median_home_value >= 300000 THEN 3
            WHEN median_home_value >= 200000 THEN 2
            WHEN median_home_value >= 100000 THEN 1
            ELSE 0
        END)
    )
    WHERE median_income IS NOT NULL;
""")

with engine.connect() as conn:
    conn.execute(query)
    conn.commit()
    
    result = conn.execute(text("""
        SELECT "GEOID", median_income, pct_vacant, 
               median_home_value, investment_score
        FROM census_tracts_delaware
        WHERE investment_score IS NOT NULL
        ORDER BY investment_score DESC
        LIMIT 5;
    """))
    
    print("🏆 En yüksek yatırım skorlu bölgeler:")
    for row in result:
        print(row)

In [ ]:
import geopandas as gpd

query = """
    SELECT "GEOID", median_income, median_home_value, 
           pct_vacant, population, investment_score, geometry
    FROM census_tracts_delaware
    WHERE investment_score IS NOT NULL
"""

gdf_final = gpd.read_postgis(query, engine, geom_col="geometry")

print(f"✅ {len(gdf_final)} tract çekildi")
print(gdf_final[["GEOID", "investment_score"]].head(3))

# GeoJSON olarak kaydet
gdf_final.to_file("delaware_investment_scores.geojson", driver="GeoJSON")
print("✅ GeoJSON kaydedildi!")

In [ ]:
import os
print(os.getcwd())